# 04 — Per-image detection-utility labels (O3, part 2)

**Settings:** Accelerator **GPU T4 x2**, Internet **On**.
**Inputs (Add Input):**
1. *Underwater Domain in ODverse33* (skycol)
2. **Your Work → notebook 03** (the committed version with the trained detector). Cell 1 copies its `results/` (splits + `best.pt`) into `/kaggle/working/results`.

For every held-out image: resize to 1280 px (long side), apply each enhancer in memory, run the frozen detector, and score per-image AP / AP50 / F1. `raw` is the baseline; `null_jpeg95` (an invisible JPEG re-encode) measures label noise. All detections are saved so selective-enhancement policies can be evaluated later without re-running the detector.

**Kaggle tips**
- Re-running cell 1 always pulls the latest code from GitHub. You never need to re-import this notebook.
- `/kaggle/working` is wiped when a session ends unless you commit (**Save Version → Save & Run All**)
  or enable **Settings → Persistence → Files**.

In [ ]:
!test -d /kaggle/working/repo || git clone -q https://github.com/Banasri007/UW-detection-aware-iqa.git /kaggle/working/repo
%run /kaggle/working/repo/scripts/kaggle_setup.py detect

In [ ]:
# Official FUnIE-GAN code + PyTorch weights (28 MB, ship inside its GitHub repo)
!python scripts/get_enhancers.py

In [ ]:
# Visual sanity check: raw vs every enhancer on 3 held-out images
%run scripts/show_enhancers.py --n 3 --deep funiegan

### Smoke test (~5 min)
Check the s/img and ETA it prints, and that `raw` per-image AP looks sensible (the detector's held-out mAP50-95 was 0.556).

In [ ]:
!python scripts/build_utility_labels.py --set ruod_pred_test --limit 40 --deep funiegan --tag smoke

### Full labelling — run with **Save Version → Save & Run All (Commit)**
Shards are resumable: if a commit times out, attach that version's output as an extra input and commit again; finished shards are restored by cell 1 and skipped.

In [ ]:
# RUOD pred_train + pred_val + pred_test (7,000 images)
!python scripts/build_utility_labels.py --set ruod_pred --deep funiegan

In [ ]:
# Duplicate-free DUO (4,539 images), cross-dataset labels for O5
!python scripts/build_utility_labels.py --set duo_clean --deep funiegan